# M10 on Kaggle: TreeSHAP explanations of E4 (`e4_added`, D11) on the full pool

**Seed:** set `SEED` in the first code cell. Run 42 first, then 43 in a second session, as for M9. M10 reads M9's report for the same seed (K, and the scores to check against), which is in the repo.

**Attach as input:** the same private Kaggle dataset as `m3_kaggle.ipynb` (the five files of the M1 `processed/` folder).

**Settings:** Internet *On*. Accelerator: *None*.
**Run:** *Save Version → Save & Run All (Commit)*.
Time is an estimate, not measured: the refit of `e4_added` took about 80 min in M9, and the explanations should take well under an hour, so allow 2 to 3 hours per seed.

Two steps, each in its own process: `fit` reruns `e4_added` and saves each window's models (kept on Kaggle, not zipped, as they are large), then `shap` explains them. Every step prints an exit code: 137 means the process was killed (usually out of memory).

**Output:** `/kaggle/working/m10_full_seed42_outputs.zip` (the name follows `SEED`) contains `reports/m10/full/` and the run folders. Download it and bring it back, also after a failure.


In [ ]:
SEED = 42
TAG = "" if SEED == 42 else f"seed{SEED}"   # seed 42 is the main report, as in M7
TAG_ARG = f"--tag {TAG}" if TAG else ""
import os, subprocess, sys
REPO = "/kaggle/working/IDS-code"
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
# --no-deps: keep Kaggle's preinstalled stack intact; versions are recorded per run.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
os.chdir(REPO)
os.environ["IDS_REPO_ROOT"] = REPO
print(subprocess.run(["git", "log", "-1", "--oneline"], capture_output=True, text=True).stdout)
assert os.path.exists("scripts/m10_shap.py"), "This clone predates M10 (no scripts/m10_shap.py). Restart the session so it clones the latest main."

In [ ]:
# Rebuild the M2 partition for the full pool (deterministic, ~1 min)
!python scripts/m2_build_partitions.py --env kaggle --pools full
# M2 redraws these figures, and their bytes differ on Kaggle. Nothing reads them. A changed report or JSON stays flagged.
!git checkout -- 'reports/m2/full/*.png'


In [ ]:
import glob, os, shutil
REP = os.path.join("reports/m10/full", TAG)
def pack():
    stage = "/kaggle/working/m10_stage"
    shutil.rmtree(stage, ignore_errors=True)
    os.makedirs(stage)    # an empty zip still comes back when a step wrote nothing
    if os.path.isdir(REP):
        shutil.copytree(REP, f"{stage}/{REP}", ignore=shutil.ignore_patterns("checkpoints"))
    for r in sorted(glob.glob("/kaggle/working/results/runs/*_m10_shap_full")):
        shutil.copytree(r, f"{stage}/results/runs/{os.path.basename(r)}")
    shutil.make_archive(f"/kaggle/working/m10_full_seed{SEED}_outputs", "zip", stage)
    shutil.rmtree(stage)
    print("zip written:", sorted(os.listdir(REP)) if os.path.isdir(REP) else [])


In [ ]:
# fit, then shap, each in its own process. 4 threads, as in M9, so the refit can match M9's scores
for step in ["fit", "shap"]:
    print("=====", step, flush=True)
    !python -X faulthandler scripts/m10_shap.py --env kaggle --pool full --set resources.n_jobs=4 seed={SEED} {TAG_ARG} --only {step}; echo "exit code: $?"
    pack()
    if not os.path.exists(f"{REP}/fit_check.json"):
        break    # no models to explain
if os.path.exists(f"{REP}/SHAP.md"):
    print(open(f"{REP}/SHAP.md", encoding="utf-8").read()[:6000])
